In [ ]:
%%bash 

curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
%%bash

uv tool install crewai crewai_tools

In [ ]:
%pip install -U 'crewai[tools, litellm]'

##### Run ollama locally

In [ ]:
%%bash 
curl -fsSL https://ollama.com/install.sh | sh
#
ollama list

#
ollama run llama3 
ollama stop llama3

#
sudo systemctl start ollama
sudo systemctl stop ollama

#
curl -X POST http://localhost:11434/api/generate -d '{
  "model": "llama3",
  "prompt":"Why is the sky blue?"
}'

# ollama run deepseek-r1:8b
# ollama run deepseek-r1:14b
# ollama run deepseek-r1:32b
# ollama run hf.co/brijeshdhaker/gpt-oss-20b-GGUF:Q4_K_M
# http://localhost:11434
# gpt-oss-120b
ollama pull gpt-oss:120b
ollama run gpt-oss:120b

ollama rm llama3

curl -X DELETE http://localhost:11434/api/delete -d '{ "model": "gemma3" }'

#### Seup Docker Model Runner Locally

In [ ]:
%%bash

sudo -S apt-get update
sudo -S apt-get install docker-model-plugin

%%bash

docker model pull ai/smollm2:360M-Q4_K_M
docker model pull hf.co/brijeshdhaker/Llama-3.2-1B-Instruct-GGUF

In [ ]:
%%bash

docker model run --detach ai/smollm2
docker model run hf.co/openai/gpt-oss-120b
docker model run hf.co/openai/gpt-oss-20b

docker model list
docker model rm hf.co/openai/gpt-oss-120b
docker model purge
docker model uninstall-runner --models
docker model uninstall-runner --images

In [ ]:
%mkdir -p agentic-ai/crewai/config

```yaml
# ./docker-compose.yaml
services:
  app:
    image: my-app
    models:
      llm:
        endpoint_var: AI_MODEL_URL
        model_var: AI_MODEL_NAME
      embedding-model:
        endpoint_var: EMBEDDING_URL
        model_var: EMBEDDING_NAME

models:
  dev_model:
    model: ai/model
    context_size: 4096
    runtime_flags:
  llm:
    model: ai/smollm2
    context_size: 4096
    runtime_flags:
      - "--temp 0.1"            # Temperature
      - "--top-p 0.9"           # Top-p sampling
      - "--verbose"             # Set verbosity level to infinity
      - "--verbose-prompt"      # Print a verbose prompt before generation
      - "--log-prefix"          # Enable prefix in log messages
      - "--log-timestamps"      # Enable timestamps in log messages
      - "--log-colors"          # Enable colored logging
  embedding-model:
    model: ai/all-minilm
    context_size: 2048
    runtime_flags:
      - "--embeddings"
```

In [ ]:
%%bash

pip install crewai crewai_tools langchain langchain_community langchain_ollama streamlit duckduckgo-search

In [ ]:
%%bash

cat <<EOF > agentic-ai/crewai/config/agents.yaml
researcher:
  role: >
    {topic} Senior Data Researcher
  goal: >
    Uncover cutting-edge developments in {topic}
  backstory: >
    You're a seasoned researcher with a knack for uncovering the latest
    developments in {topic}. You find the most relevant information and
    present it clearly.
EOF


In [ ]:
%%bash

cat <<EOF > agentic-ai/crewai/config/tasks.yaml
research_task:
  description: >
    Conduct thorough research about {topic}. Use web search to find current,
    credible information. The current year is 2026.
  expected_output: >
    A markdown report with clear sections: key trends, notable tools or companies,
    and implications. Aim for 800–1200 words. No fenced code blocks around the whole document.
  agent: researcher
  output_file: output/report.md
EOF

In [ ]:
# src/latest_ai_flow/crews/content_crew/content_crew.py
from typing import List

from crewai import Agent, Crew, Process, Task
from crewai.agents.agent_builder.base_agent import BaseAgent
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool


@CrewBase
class ResearchCrew:
  """Single-agent research crew used inside the Flow."""

  agents: List[BaseAgent]
  tasks: List[Task]

  agents_config = "agentic-ai/crewai/config/agents.yaml"
  tasks_config = "agentic-ai/crewai/config/tasks.yaml"

  @agent
  def researcher(self) -> Agent:
    return Agent(
      config=self.agents_config["researcher"],  # type: ignore[index]
      verbose=True,
      tools=[SerperDevTool()],
    )

  @task
  def research_task(self) -> Task:
    return Task(
      config=self.tasks_config["research_task"],  # type: ignore[index]
    )

  @crew
  def crew(self) -> Crew:
    return Crew(
      agents=self.agents,
      tasks=self.tasks,
      process=Process.sequential,
      verbose=True,
    )

In [ ]:
# src/latest_ai_flow/main.py
from pydantic import BaseModel
from crewai.flow import Flow, listen, start
#from latest_ai_flow.crews.content_crew.content_crew import ResearchCrew


class ResearchFlowState(BaseModel):
  topic: str = ""
  report: str = ""


class LatestAiFlow(Flow[ResearchFlowState]):
  @start()
  def prepare_topic(self, crewai_trigger_payload: dict | None = None):
    if crewai_trigger_payload:
      self.state.topic = crewai_trigger_payload.get("topic", "AI Agents")
    else:
      self.state.topic = "AI Agents"
    print(f"Topic: {self.state.topic}")

  @listen(prepare_topic)
  def run_research(self):
    result = ResearchCrew().crew().kickoff(inputs={"topic": self.state.topic})
    self.state.report = result.raw
    print("Research crew finished.")

  @listen(run_research)
  def summarize(self):
    print("Report path: output/report.md")


def kickoff():
  LatestAiFlow().kickoff()


def plot():
  LatestAiFlow().plot()


#if __name__ == "__main__":
#  kickoff()

In [ ]:
kickoff()